# Aivora: Online Shopping Support Chatbot

Aivora is an ecommerce support chatbot for orders, returns, delivery, payments, accounts, products, offers, memberships, sellers, support escalation, and complaint handling.

## Project Summary

The chatbot follows the starter-template stages: Part 1 builds the conversation loop, Part 2 performs pattern matching, Part 3 loads chatbot knowledge from JSON, and Part 4 adds NLTK preprocessing with extensions such as synonyms, typo handling, sentiment detection, priority detection, memory, context follow-ups, and ticket summaries.

## Setup

This section imports the libraries used by the chatbot. Standard Python libraries handle JSON files, regular expressions, text formatting, and random response selection.

NLTK is used for tokenising words, stemming, and checking small spelling mistakes in user input.


In [ ]:
# Imports standard tools and NLTK helpers used across the chatbot.
from copy import deepcopy
import json
import random
import re
import string
import textwrap

try:
    import nltk
    from nltk.metrics.distance import edit_distance
    from nltk.stem import PorterStemmer
    from nltk.tokenize import wordpunct_tokenize
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        "NLTK is required for Aivora. Install it in this Jupyter kernel with: python -m pip install nltk"
    ) from error

In [ ]:
stemmer = PorterStemmer()

# Stores the details Aivora remembers during one chat session.
initial_state = {
    "name": "",
    "preference": "",
    "order_number": "",
    "last_issue": "",
    "last_topic": "",
    "pending_slot": "",
    "pending_intent": "",
    "issue_history": [],
    "priority": False,
    "priority_count": 0
}

user_state = deepcopy(initial_state)

## Part 1: Conversation Loop

This part creates the main chat loop. The chatbot welcomes the user, accepts repeated input, and ends the conversation when the user types `exit` or `quit`.

The loop also keeps the output readable by formatting Aivora's replies neatly in the notebook.


In [ ]:
# Prints Aivora replies in the notebook.
def print_aivora(message):
    print(textwrap.fill(
        message,
        width=46,
        initial_indent="Aivora: ",
        subsequent_indent="        ",
        break_long_words=False,
        break_on_hyphens=False
    ))

In [ ]:
# Clears memory so every new chat starts fresh.
def reset_chat_state():
    user_state.clear()
    user_state.update(deepcopy(initial_state))

In [ ]:
# Runs the conversation loop until the user exits.
def chatbot():
    reset_chat_state()
    knowledge = chatbot_knowledge
    messages = knowledge["messages"]

    for message in messages["welcome"]:
        print_aivora(message)

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() in {"exit", "quit"}:
            print_aivora(build_ticket_summary(messages))
            print_aivora(messages["goodbye"])
            break

        if not user_input:
            print_aivora(messages["empty_input"])
            continue

        print_aivora(chatbot_response(user_input, knowledge))

## Part 2: Pattern Matching

This part detects what the user is asking for. It uses regular expressions, keyword scoring, memory checks, and conversation context to choose the most suitable intent.

The logic is kept separate from the response data so the chatbot can be improved by updating the JSON file.


In [ ]:
# Prevents missing template values from breaking a response.
class SafeFormatDict(dict):
    def __missing__(self, key):
        return ""

In [ ]:
# Prepares remembered values for response templates.
def format_values():
    name = user_state["name"]

    return SafeFormatDict({
        "name": name,
        "name_suffix": f", {name}" if name else "",
        "preference": user_state["preference"],
        "order_number": user_state["order_number"],
        "last_issue": user_state["last_issue"]
    })

In [ ]:
# Returns the first matched group from a regex rule.
def first_regex_group(match, group_numbers):
    for group_number in group_numbers:
        value = match.group(group_number)
        if value:
            return value
    return ""

In [ ]:
# Builds a lookup so synonyms can map back to one main word.
def build_synonym_lookup(groups):
    lookup = {}

    for main_word, alternatives in groups.items():
        lookup[main_word] = main_word
        for word in alternatives:
            lookup[word] = main_word

    return lookup

In [ ]:
# Converts JSON language settings into sets and dictionaries for faster lookup.
def configure_language_tools(language_data):
    return {
        "stop_words": set(language_data["stop_words"]),
        "synonym_groups": language_data["synonym_groups"],
        "typo_corrections": dict(language_data["typo_corrections"]),
        "correction_vocabulary": set(language_data["correction_vocabulary"]),
        "negative_words": set(language_data["sentiment_words"]["negative"]),
        "positive_words": set(language_data["sentiment_words"]["positive"]),
        "priority_words": set(language_data["priority_words"]),
        "priority_phrases": list(language_data["priority_phrases"]),
        "order_followups": {
            intent: set(words)
            for intent, words in language_data["order_followups"].items()
        },
        "clarify_topic_keywords": {
            intent: set(words)
            for intent, words in language_data["clarify_topic_keywords"].items()
        },
        "direct_menu_words": set(language_data["direct_menu_words"]),
        "problem_words": set(language_data["problem_words"]),
        "order_reference_words": set(language_data["order_reference_words"])
    }

In [ ]:
# Splits user input into lowercase word tokens using NLTK.
def tokenize(text):
    tokens = wordpunct_tokenize(text)
    return [token.lower() for token in tokens if re.search(r"\w", token)]

In [ ]:
# Corrects known typos and close spelling mistakes.
def correct_token(token):
    typo_corrections = language_tools["typo_corrections"]
    correction_vocabulary = language_tools["correction_vocabulary"]

    if token in typo_corrections:
        return typo_corrections[token]
    if token in correction_vocabulary or len(token) < 4 or token.isdigit():
        return token

    candidates = [
        word for word in correction_vocabulary
        if token[0] == word[0] and abs(len(token) - len(word)) <= 2
    ]
    if not candidates:
        return token

    best_match = min(candidates, key=lambda word: edit_distance(token, word))
    distance = edit_distance(token, best_match)

    return best_match if distance <= 2 else token

In [ ]:
# Creates standard and stemmed forms of a word for matching.
def nltk_word_forms(word):
    corrected = correct_token(word)
    canonical = synonym_lookup.get(corrected, corrected)
    stemmed = stemmer.stem(canonical)

    forms = [canonical, stemmed]
    return list(dict.fromkeys(forms))

In [ ]:
# Cleans and normalises user text before intent scoring.
def preprocess(text):
    cleaned = text.lower().translate(str.maketrans("", "", string.punctuation.replace("#", "")))
    processed_words = []

    for token in tokenize(cleaned):
        if token in language_tools["stop_words"]:
            continue
        processed_words.extend(nltk_word_forms(token))

    return " ".join(dict.fromkeys(processed_words))

In [ ]:
# Saves user details such as name, preference and order number.
def update_memory(user_input, knowledge):
    patterns = knowledge["memory_patterns"]

    name_rule = patterns["name"]
    name_match = re.search(name_rule["pattern"], user_input, flags=re.IGNORECASE)
    if name_match:
        user_state["name"] = first_regex_group(name_match, name_rule["groups"]).title()
        return "name"

    preference_rule = patterns["preference"]
    preference_match = re.search(preference_rule["pattern"], user_input, flags=re.IGNORECASE)
    if preference_match:
        user_state["preference"] = first_regex_group(preference_match, preference_rule["groups"]).strip().lower()
        return "preference"

    order_rule = patterns["order_number"]
    bare_order_rule = patterns["bare_order_number"]
    corrected_order_rule = patterns["corrected_order_number"]

    order_match = re.search(order_rule["pattern"], user_input, flags=re.IGNORECASE)
    bare_order_match = re.search(bare_order_rule["pattern"], user_input.strip())
    corrected_tokens = [correct_token(token) for token in tokenize(user_input)]
    corrected_order_match = None

    if corrected_order_rule["trigger_word"] in corrected_tokens:
        corrected_order_match = re.search(corrected_order_rule["pattern"], user_input)

    if order_match or bare_order_match or corrected_order_match:
        if order_match:
            user_state["order_number"] = first_regex_group(order_match, order_rule["groups"])
        elif bare_order_match:
            user_state["order_number"] = first_regex_group(bare_order_match, bare_order_rule["groups"])
        else:
            user_state["order_number"] = first_regex_group(corrected_order_match, corrected_order_rule["groups"])

        topic = user_state["pending_intent"] or user_state["last_topic"]
        user_state["pending_slot"] = ""
        user_state["pending_intent"] = ""

        if topic == "order_status":
            return "order_number_tracking"
        if topic == "returns":
            return "order_number_return"
        if topic == "delivery":
            return "order_number_delivery"
        if topic == "payment":
            return "order_number_payment"
        return "order_number"

    return None

In [ ]:
# Checks whether a JSON follow-up rule matches the user input.
def rule_matches(text, words, rule):
    keyword_match = words.intersection(set(rule.get("keywords", [])))
    phrase_match = any(phrase in text for phrase in rule.get("phrases", []))
    return bool(keyword_match or phrase_match)

In [ ]:
# Detects urgent words or phrases that should be prioritised.
def detect_priority(text):
    words = set(tokenize(text)).union(set(preprocess(text).split()))
    text_lower = text.lower()

    if words.intersection(language_tools["priority_words"]):
        return True
    return any(phrase in text_lower for phrase in language_tools["priority_phrases"])

In [ ]:
# Detects negative wording so Aivora can respond politely.
def detect_sentiment(text):
    words = set(tokenize(text))
    processed_words = set(preprocess(text).split())
    all_words = words.union(processed_words)

    return "negative" if all_words.intersection(language_tools["negative_words"]) else "neutral"

In [ ]:
# Handles follow-up requests about an order already in memory.
def detect_order_followup(text):
    if not user_state["order_number"]:
        return None

    words = set(tokenize(text)).union(set(preprocess(text).split()))
    if not words.intersection(language_tools["order_reference_words"]):
        return None

    for intent, keywords in language_tools["order_followups"].items():
        if words.intersection(keywords):
            return intent

    return None

In [ ]:
# Uses the pending support slot to keep replies on topic.
def detect_slot_followup(text, knowledge):
    slot = user_state["pending_slot"]
    if not slot:
        return None

    words = set(tokenize(text)).union(set(preprocess(text).split()))
    text_lower = text.lower()

    for rule in knowledge["slot_followups"].get(slot, []):
        if rule_matches(text_lower, words, rule):
            return rule["intent"]

    return None

In [ ]:
# Uses the last topic to handle natural follow-up messages.
def detect_context_followup(text, knowledge):
    topic = user_state["last_topic"]
    if not topic:
        return None

    words = set(tokenize(text)).union(set(preprocess(text).split()))
    text_lower = text.lower()

    for rule in knowledge["context_followups"].get(topic, []):
        if rule_matches(text_lower, words, rule):
            return rule["intent"]

    if topic == "clarify_issue":
        for intent, keywords in language_tools["clarify_topic_keywords"].items():
            if words.intersection(keywords):
                return intent

    return None

In [ ]:
# Asks for clarification when the issue is too general.
def detect_vague_topic(text):
    words = set(tokenize(text)).union(set(preprocess(text).split()))

    if text.lower().strip() in language_tools["direct_menu_words"]:
        return None
    if not words.intersection(language_tools["problem_words"]):
        return None

    for intent, keywords in language_tools["clarify_topic_keywords"].items():
        if words.intersection(keywords):
            return intent

    return "clarify_issue"

In [ ]:
# Scores regex and keyword matches to choose the strongest intent.
def score_intents(user_input, knowledge):
    original_text = user_input.lower().strip()
    processed_text = preprocess(user_input)
    raw_tokens = tokenize(user_input)
    words = set(raw_tokens).union(set(processed_text.split()))
    corrected_words = {
        correct_token(token) for token in raw_tokens
        if correct_token(token) != token
    }
    scores = {}

    for pattern, intent in knowledge["compiled_patterns"]:
        if intent in {"default", "priority_issue"}:
            continue
        if intent.startswith("followup_") and not user_state["order_number"]:
            continue
        if pattern.search(original_text):
            scores[intent] = scores.get(intent, 0) + 4
        elif processed_text and pattern.search(processed_text):
            scores[intent] = scores.get(intent, 0) + 2

    for intent, keywords in knowledge["confidence_keywords"].items():
        if intent.startswith("followup_") and not user_state["order_number"]:
            continue
        keyword_set = set(keywords)
        overlap = words.intersection(keyword_set)
        corrected_overlap = corrected_words.intersection(keyword_set)
        if overlap:
            scores[intent] = scores.get(intent, 0) + len(overlap)
        if corrected_overlap:
            scores[intent] = scores.get(intent, 0) + (2 * len(corrected_overlap))

    if not scores:
        return "default", 0

    intent_order = {intent: index for index, (_, intent) in enumerate(knowledge["compiled_patterns"])}
    best_intent, best_score = max(
        scores.items(),
        key=lambda item: (item[1], -intent_order.get(item[0], 9999))
    )

    if best_score < knowledge["confidence_threshold"]:
        return "default", best_score
    return best_intent, best_score

In [ ]:
# Returns the intent selected by the scoring function.
def match_regex_intent(user_input, knowledge):
    intent, score = score_intents(user_input, knowledge)
    return intent

In [ ]:
# Checks memory, follow-ups, context, and intent matching in priority order.
def choose_intent(user_input, knowledge):
    memory_intent = update_memory(user_input, knowledge)
    if memory_intent:
        return memory_intent

    order_followup_intent = detect_order_followup(user_input)
    if order_followup_intent:
        return order_followup_intent

    slot_intent = detect_slot_followup(user_input, knowledge)
    if slot_intent:
        return slot_intent

    context_intent = detect_context_followup(user_input, knowledge)
    if context_intent:
        return context_intent

    matched_intent = match_regex_intent(user_input, knowledge)
    if matched_intent != "default":
        return matched_intent

    vague_topic_intent = detect_vague_topic(user_input)
    if vague_topic_intent:
        return vague_topic_intent

    if detect_priority(user_input):
        return "priority_issue"

    return "default"

In [ ]:
# Public pattern-detection wrapper used by the response function.
def detect_pattern(user_input, knowledge=None):
    active_knowledge = knowledge or chatbot_knowledge
    return choose_intent(user_input, active_knowledge)

## Part 3: File Handling With JSON

This part loads chatbot knowledge from `intents.json`. The JSON file stores patterns, responses, next steps, memory rules, synonyms, priority words, and support-flow rules.

Using a JSON file makes the chatbot data-driven instead of hardcoding responses and dictionaries inside the notebook.


In [ ]:
# Reads the JSON file that stores chatbot knowledge.
def read_json(file_path="intents.json"):
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

In [ ]:
# Converts JSON regex patterns into compiled intent mappings.
def store_patterns(intents_json):
    pattern2intent = {}
    compiled_patterns = []

    for intent in intents_json["intents"]:
        tag = intent["tag"]
        for pattern in intent["patterns"]:
            pattern2intent[pattern] = tag
            compiled_patterns.append((re.compile(pattern, re.IGNORECASE), tag))

    return pattern2intent, compiled_patterns

In [ ]:
# Stores responses, next steps, and summary labels by intent.
def store_responses(intents_json):
    intent2responses = {}
    next_step_by_intent = {}
    summary_by_intent = {}

    for intent in intents_json["intents"]:
        tag = intent["tag"]
        intent2responses[tag] = intent["responses"]
        next_step_by_intent[tag] = intent.get("next_step", "")
        summary_by_intent[tag] = intent.get("summary", "")

    return intent2responses, next_step_by_intent, summary_by_intent

In [ ]:
# Loads all chatbot patterns, responses, and support rules from the JSON file.
def load_intents(file_path="intents.json"):
    data = read_json(file_path)
    pattern2intent, compiled_patterns = store_patterns(data)
    intent2responses, next_step_by_intent, summary_by_intent = store_responses(data)

    return {
        "pattern2intent": pattern2intent,
        "compiled_patterns": compiled_patterns,
        "intent2responses": intent2responses,
        "next_step_by_intent": next_step_by_intent,
        "summary_by_intent": summary_by_intent,
        "summary_replacements": data["summary_replacements"],
        "confidence_keywords": data["confidence_keywords"],
        "confidence_threshold": data["confidence_threshold"],
        "slot_rules": data["slot_rules"],
        "slot_followups": data["slot_followups"],
        "context_followups": data["context_followups"],
        "messages": data["messages"],
        "memory_patterns": data["memory_patterns"],
        "language": data["language"],
        "topic_map": data["topic_map"],
        "completed_slot_intents": data["completed_slot_intents"],
        "single_response_intents": data["single_response_intents"]
    }

In [ ]:
# Initialises the JSON knowledge base and language tools.
chatbot_knowledge = load_intents()
language_tools = configure_language_tools(chatbot_knowledge["language"])
synonym_lookup = build_synonym_lookup(language_tools["synonym_groups"])

## Part 4: NLTK And Extensions

This part adds the advanced chatbot features. It uses NLTK preprocessing, synonym handling, typo correction, sentiment detection, priority detection, context follow-ups, and ticket summaries.

These extensions make the chatbot respond more naturally while still following the support topics stored in JSON.


In [ ]:
# Updates the chat summary after each handled issue.
def update_issue_summary(intent, knowledge, is_priority=False):
    if is_priority:
        user_state["priority"] = True
        user_state["priority_count"] += 1

    issue_label = knowledge["summary_by_intent"].get(intent, "")
    if intent == "payment" and user_state["last_issue"] == knowledge["summary_by_intent"].get("payment_duplicate", ""):
        issue_label = user_state["last_issue"]

    if issue_label:
        for generic_label, specific_labels in knowledge["summary_replacements"].items():
            if issue_label in specific_labels and generic_label in user_state["issue_history"]:
                user_state["issue_history"].remove(generic_label)

        if issue_label not in user_state["issue_history"]:
            user_state["issue_history"].append(issue_label)
        user_state["last_issue"] = issue_label

    if intent in knowledge["topic_map"]:
        user_state["last_topic"] = knowledge["topic_map"][intent]

In [ ]:
# Tracks whether Aivora is waiting for a specific follow-up detail.
def update_pending_slot(intent, knowledge):
    slot_rule = knowledge["slot_rules"].get(intent)
    if slot_rule:
        user_state["pending_slot"] = slot_rule["slot"]
        user_state["pending_intent"] = intent
        return

    if intent in set(knowledge["completed_slot_intents"]):
        user_state["pending_slot"] = ""
        user_state["pending_intent"] = ""

In [ ]:
# Gets the next-step guidance stored for an intent.
def recommended_next_step(intent, knowledge):
    return knowledge["next_step_by_intent"].get(intent, "")

In [ ]:
# Chooses a suitable response template for the detected intent.
def select_response(intent, knowledge):
    response_map = knowledge["intent2responses"]
    responses = response_map.get(intent, response_map["default"])

    if intent in set(knowledge["single_response_intents"]):
        return responses[0]

    return random.choice(responses)

In [ ]:
# Builds the final chatbot reply using the detected intent and JSON response data.
def generate_response(user_input, knowledge):
    current_priority = detect_priority(user_input)
    intent = detect_pattern(user_input, knowledge)
    update_issue_summary(intent, knowledge, current_priority)
    update_pending_slot(intent, knowledge)

    response_template = select_response(intent, knowledge)
    response = response_template.format_map(format_values())

    messages = knowledge["messages"]
    sentiment = detect_sentiment(user_input)
    if sentiment == "negative" and intent not in {"priority_issue", "upset_customer"}:
        response = messages["negative_prefix"] + response

    next_step = recommended_next_step(intent, knowledge)
    if next_step:
        response = f"{response} {next_step}"

    if current_priority and intent not in {"priority_issue", "upset_customer"}:
        response = messages["priority_prefix"] + response

    return response

In [ ]:
# Small wrapper that returns Aivora response text for user input.
def chatbot_response(user_input, knowledge=None):
    active_knowledge = knowledge or chatbot_knowledge
    return generate_response(user_input, active_knowledge)

In [ ]:
# Creates a short summary of the issue handled during the chat.
def build_ticket_summary(messages):
    details = []

    if user_state["name"]:
        details.append(f"{messages['name_label']}: {user_state['name']}")
    if user_state["order_number"]:
        details.append(f"{messages['order_label']}: {user_state['order_number']}")

    issues = user_state["issue_history"]
    if issues:
        label = messages["single_issue_label"] if len(issues) == 1 else messages["multiple_issues_label"]
        details.append(f"{label}: {'; '.join(issues)}")
    elif user_state["last_issue"]:
        details.append(f"{messages['single_issue_label']}: {user_state['last_issue']}")

    if user_state["preference"]:
        details.append(f"{messages['preference_label']}: {user_state['preference']}")
    if user_state["priority_count"]:
        label = messages["priority_single_label"] if user_state["priority_count"] == 1 else messages["priority_multiple_label"]
        details.append(f"{label}: {user_state['priority_count']}")

    if not details:
        return messages["summary_empty"]

    return messages["summary_prefix"] + ". ".join(details) + "."

## Run Aivora

Run the final cell to start the chatbot. The conversation will continue until the user types `exit` or `quit`.

The chatbot will show a short chat summary before closing.


In [ ]:
# Start Aivora.
chatbot()